# TV series popularity: exploration notes

**Task:** find what is associated with popularity for Netflix TV series, then outline an ML approach to predict popularity before launch.

**Cleaning Data:** Within this notebook I will be cleaning and understanding the data before I do start a full EDA. `raw` is the unmodified data. Nothing is cleaned until the exploration says what needs cleaning. Every cleaning step gets a row in the cleaning log at the bottom.

In [1]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [2]:
raw = pd.read_csv("../data/tmdb_tv_shows/TMDB_tv_dataset_v3.csv")


In [3]:
# print(raw.shape)
# raw.info()

In [4]:
raw.describe().T

,count,mean,std,min,25%,50%,75%,max
id,168639.0,111307.074704,76451.662352,1.0,45936.5,97734.000,196923.5000,251213.000
number_of_seasons,168639.0,1.548497,2.942872,0.0,1.0,1.000,1.0000,240.000
number_of_episodes,168639.0,24.465082,134.799622,0.0,1.0,6.000,20.0000,20839.000
vote_count,168639.0,13.305054,190.809059,0.0,0.0,0.000,1.0000,21857.000
vote_average,168639.0,2.333843,3.454334,0.0,0.0,0.000,6.0000,10.000
popularity,168639.0,5.882644,42.023216,0.0,0.6,0.857,2.4315,3707.008
episode_run_time,168639.0,22.603348,47.950427,0.0,0.0,0.000,42.0000,6032.000


**Observations**
- `name` count > uniqiue, --> May need investigating as they could be duplicates
- `ID` repeated ids
- `number_of_seasons` and `number_of_episodes` have outlier at max --> Determine if they are valid or not
- `vote_count`,`vote_average` and `popularity` --> maybe considered as key indicators
- `vote_counts` are skewed
- `overview` repeats --> duplicates?
- `genre` needs to be seperated
- `created_by` may give insight
- Placeholders within columns such as `episode_run_time`,`number_of_episodes` and `popularity`
- `votes` can be considered as unreliable
- convert columsn to there respective format

In [5]:
SUFFIX = re.compile(
    r"(inc\.?|llc|l\.l\.c\.?|ltd\.?|limited|co\.?|corp\.?|corporation|gmbh|s\.a\.?|lp|l\.p\.|pty|plc|ag|bv|b\.v\.?)",
    re.I,
)

def split_list(text):
    """Split 'a, b, c' into a list, gluing legal suffixes (Inc., LLC...) back on."""
    out = []
    for part in text.split(", "):
        part = part.strip()
        if out and SUFFIX.fullmatch(part):
            out[-1] = f"{out[-1]}, {part}"
        else:
            out.append(part)
    return out

clean = raw.copy()

# blank text -> missing, so blank and NaN mean the same thing
text_cols = clean.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    clean[c] = clean[c].where(clean[c].str.strip() != "")

# delete exact duplicates
clean = clean.drop_duplicates().copy()

# normalise text
clean["name_clean"] = (
    clean.name.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
)

# lists: plain split for codes and genres, suffix-aware split where names can contain "Name, Inc."
plain_cols = ["genres", "languages", "origin_country"]
suffix_cols = ["created_by", "networks", "production_companies"]
for c in plain_cols:
    clean[c + "_list"] = clean[c].apply(
        lambda t: [x.strip() for x in t.split(", ")] if isinstance(t, str) else t
    )
for c in suffix_cols:
    clean[c + "_list"] = clean[c].apply(lambda t: split_list(t) if isinstance(t, str) else t)

# dates
for c in ["first_air_date", "last_air_date"]:
    before = clean[c].isna().sum()
    clean[c] = pd.to_datetime(clean[c], format="%Y-%m-%d", errors="coerce")
    assert clean[c].isna().sum() == before, f"{c}: some dates failed to parse"


In [6]:
# quick checks on the cleaned table
print(len(raw), len(clean), clean.id.duplicated().sum())   # expect 168639 167059 2354
print(clean.name.nunique(), clean.name_clean.nunique())     # expect 155585 154912
print([col for col in clean.columns if col.endswith("_list")])   # expect 6 columns

168639 167059 2354
155585 154912
['genres_list', 'languages_list', 'origin_country_list', 'created_by_list', 'networks_list', 'production_companies_list']


## Netflix subset

In [7]:
is_netflix = clean.networks_list.apply(
    lambda nets: isinstance(nets, list) and "netflix" in [n.lower() for n in nets]
)
nf_clean = clean[is_netflix].copy()

print(len(nf_clean), nf_clean.id.nunique())   # expect 1957 1929
nf_clean.describe(include="all").T

1957 1929


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
id,1957.0,NaN,NaN,NaN,136522.158917,1424.0,88728.0,115497.0,205186.0,250850.0,59366.897255
name,1957,1927,Till Murder Do Us Part: Soering vs. Haysom,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
number_of_seasons,1957.0,NaN,NaN,NaN,1.56464,1.0,1.0,1.0,2.0,30.0,1.506474
number_of_episodes,1957.0,NaN,NaN,NaN,14.711804,0.0,6.0,8.0,16.0,973.0,33.596437
original_language,1957,33,en,1091,NaN,NaN,NaN,NaN,NaN,NaN,NaN
vote_count,1957.0,NaN,NaN,NaN,192.044967,0.0,5.0,20.0,84.0,17836.0,891.206723
vote_average,1957.0,NaN,NaN,NaN,6.010488,0.0,5.871,6.917,7.527,10.0,2.563966
overview,1894,1872,Did Jens Soering murder his girlfriend's paren...,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
adult,1957,2,False,1955,NaN,NaN,NaN,NaN,NaN,NaN,NaN
backdrop_path,1810,1793,/toxNmhb2TpVL59OAJNd8H6XqrBE.jpg,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Cleaning log

Applied to the whole dataset (`clean`), then filtered to Netflix (`nf_clean`).

| Step | Why | Effect |
|---|---|---|
| Blank text to missing | blank and NaN should mean the same thing | blank `name` (1), `overview` (16), `homepage` (2) become NaN |
| Drop exact duplicate rows | identical in every column, no information lost | 168,639 to 167,059 rows (1,580 removed) |
| `name_clean` (strip, lowercase, collapse spaces) | match names that differ only by case or spacing | 155,585 to 154,912 distinct names. `name` is kept as it was |
| Split list columns into `_list` columns | one value per item, not one string | 6 columns: genres, languages, origin_country, created_by, networks, production_companies |
| Re-join suffixes in `created_by`, `networks`, `production_companies` | "Company, Inc." was being cut in two | 0 stray suffix items left |
| Plain split for genres, languages, origin_country | country codes like CO or AG look like suffixes | no change to those items |
| Dates to datetime | text to real dates | no parse failures (assert passes) |
| Filter to Netflix on `networks_list` | the task is about Netflix | 1,957 rows, 1,929 shows |

**[open]** Not cleaned yet:
- Repeated IDs after the exact-duplicate drop: 2,354 in the full data, 28 in the Netflix subset. Needs a rule for which copy to keep.
- Zero placeholders: `episode_run_time` is 0 in many rows, `number_of_episodes` and `popularity` in a few.
- Outliers in `number_of_seasons` and `number_of_episodes`: valid or not?
- 3 names in the Netflix subset are shared by different IDs: same show or different?